# Dynamic Programming Foundations: Fibonacci, Memoization, and Tabulation

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. The Core Philosophy of Dynamic Programming

Dynamic Programming (DP) is an algorithm design technique for solving optimization and recurrence problems characterized by two properties:
1. **Optimal Substructure:** An optimal solution to the problem contains within it optimal solutions to subproblems.
2. **Overlapping Subproblems:** A recursive algorithm visits the *same* subproblems repeatedly rather than generating new subproblems.

### Divide-and-Conquer vs Dynamic Programming
- **Divide-and-Conquer:** Subproblems are disjoint and independent (e.g., left and right halves in Merge Sort). Each subproblem is solved once.
- **Dynamic Programming:** Subproblems heavily overlap (e.g., $F(n-1)$ and $F(n-2)$ both need $F(n-3)$). Solving each subproblem only once and caching the result reduces exponential time to polynomial time.


In [ ]:
from __future__ import annotations

import functools
import time

## 2. The Exponential Barrier of Naive Recursion

The Fibonacci sequence is defined by:
$$F(0) = 0, \quad F(1) = 1, \quad F(n) = F(n-1) + F(n-2) \quad (n \ge 2)$$

Evaluating this recurrence via naive recursion yields a binary call tree with $O(2^n)$ / $\Theta(\phi^n)$ function calls, where $\phi = \frac{1 + \sqrt{5}}{2} \approx 1.618$.


In [ ]:
call_count = 0


def fibonacci_naive(n: int) -> int:
    """Naive exponential recursion O(2^n)."""
    global call_count
    call_count += 1
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci_naive(n - 1) + fibonacci_naive(n - 2)


for test_n in [5, 10, 15, 20]:
    call_count = 0
    val = fibonacci_naive(test_n)
    print(f"fibonacci_naive({test_n:2d}) = {val:5d} | Recursive invocations: {call_count:6d}")

## 3. Top-Down Dynamic Programming (Memoization)

Top-down DP preserves the natural recursive formulation but maintains an auxiliary cache (`memo`). Before evaluating a subproblem, the algorithm inspects the cache; if present, it returns the stored result in $O(1)$ time.


In [ ]:
memo_calls = 0


def fibonacci_memo(n: int, memo: dict[int, int] | None = None) -> int:
    """Top-down memoized DP O(n) time and O(n) stack/cache space."""
    global memo_calls
    memo_calls += 1
    if memo is None:
        memo = {0: 0, 1: 1}
    if n in memo:
        return memo[n]

    memo[n] = fibonacci_memo(n - 1, memo) + fibonacci_memo(n - 2, memo)
    return memo[n]


# Alternatively using Python standard library lru_cache
@functools.cache
def fibonacci_lru(n: int) -> int:
    if n < 2:
        return n
    return fibonacci_lru(n - 1) + fibonacci_lru(n - 2)


memo_calls = 0
val = fibonacci_memo(20)
print(f"fibonacci_memo(20) = {val} | Total calls: {memo_calls} (Linear O(n) calls!)")
assert val == fibonacci_lru(20)

## 4. Bottom-Up Dynamic Programming (Tabulation)

Bottom-up DP iteratively fills an explicit table from the base cases upward, avoiding call-stack overhead.


In [ ]:
def fibonacci_dp(n: int) -> tuple[int, list[int]]:
    """Bottom-up DP table filling matching lecture slide implementation."""
    if n == 0:
        return 0, [0]
    if n == 1:
        return 1, [0, 1]

    # create empty dp list (table)
    dp = [0] * (n + 1)
    dp[0] = 0
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]

    return dp[n], dp


res4, table4 = fibonacci_dp(4)
print(f"F(4) = {res4}, Table: {table4}")
res7, table7 = fibonacci_dp(7)
print(f"F(7) = {res7}, Table: {table7}")
assert res4 == 3
assert res7 == 13

## 5. Space Optimization: $O(1)$ Auxiliary Space

Because $F(n)$ only depends on the two immediately preceding values $F(n-1)$ and $F(n-2)$, we do not need to store the entire array of size $n+1$. Storing just two registers achieves $O(1)$ auxiliary space.


In [ ]:
def fibonacci_optimized(n: int) -> int:
    """Space-optimized iterative DP: O(n) time, O(1) space."""
    if n == 0:
        return 0
    if n == 1:
        return 1

    prev2, prev1 = 0, 1
    for _ in range(2, n + 1):
        current = prev1 + prev2
        prev2, prev1 = prev1, current

    return prev1


assert fibonacci_optimized(50) == 12586269025
print("F(50) computed in O(1) space:", fibonacci_optimized(50))

## 6. Runtime Comparison Benchmark

We empirically verify the transition from exponential time $O(2^n)$ to linear time $O(n)$.


In [ ]:
def benchmark():
    n = 30
    t0 = time.perf_counter()
    r_naive = fibonacci_naive(n)
    t_naive = time.perf_counter() - t0

    t0 = time.perf_counter()
    r_memo = fibonacci_memo(n)
    t_memo = time.perf_counter() - t0

    t0 = time.perf_counter()
    r_dp, _ = fibonacci_dp(n)
    t_dp = time.perf_counter() - t0

    t0 = time.perf_counter()
    r_opt = fibonacci_optimized(n)
    t_opt = time.perf_counter() - t0

    print(f"N = {n}:")
    print(f"  Naive Recursive:  {t_naive:.6f} s ({r_naive})")
    print(f"  Top-Down Memo:    {t_memo:.6f} s ({r_memo})")
    print(f"  Bottom-Up Table:  {t_dp:.6f} s ({r_dp})")
    print(f"  Space Optimized:  {t_opt:.6f} s ({r_opt})")


benchmark()